# June–July temperature anomalies in Central and Eastern Europe

This notebook reproduces the figures in the Reuters analysis of June and July daily high temperatures in Hungary, Romania and Poland.

Each annual value is the country-wide average daily maximum temperature from June 1 through July 31, minus that country's 1961–1990 average for the same 61 calendar days. June contributes 30 days and July 31 days.

In [1]:
from pathlib import Path

import altair as alt
import pandas as pd

DATA_PATH = Path("data/annual-june-july-tmax-anomalies.csv")
DECADE_START = 2017
DECADE_END = 2026

In [2]:
annual = pd.read_csv(DATA_PATH).melt(
    id_vars="year",
    var_name="country",
    value_name="anomaly_degC",
)
annual.head()

,year,country,anomaly_degC
0,1961,Hungary,1.101
1,1962,Hungary,-1.087
2,1963,Hungary,2.654
3,1964,Hungary,2.367
4,1965,Hungary,-1.041


In [3]:
decade = annual.loc[annual["year"].between(DECADE_START, DECADE_END)]
decade_average = (
    decade.groupby("country", as_index=False)["anomaly_degC"]
    .mean()
    .assign(
        anomaly_degC=lambda frame: frame["anomaly_degC"].round(1),
        anomaly_degF=lambda frame: (frame["anomaly_degC"] * 9 / 5).round(1),
    )
    .sort_values("anomaly_degC", ascending=False, ignore_index=True)
)
decade_average

,country,anomaly_degC,anomaly_degF
0,Hungary,3.1,5.6
1,Romania,3.0,5.4
2,Poland,2.8,5.0


In [4]:
alt.Chart(annual).mark_line().encode(
    x=alt.X("year:Q", title="Year"),
    y=alt.Y("anomaly_degC:Q", title="Degrees Celsius above or below the 1961–1990 normal"),
    color=alt.Color("country:N", title="Country"),
    tooltip=[
        alt.Tooltip("country:N", title="Country"),
        alt.Tooltip("year:Q", title="Year"),
        alt.Tooltip("anomaly_degC:Q", title="Anomaly", format=".2f"),
    ],
).properties(
    title="June–July average daily high temperature anomaly",
    width=700,
    height=400,
)

alt.Chart(...)